# Paginación en Django Rest Framework

En este ejercicio se implementó paginación para la API del eCommerce utilizando `PageNumberPagination`.

## Clase de paginación

Se creó una clase llamada `ProductPagination` que hereda de `PageNumberPagination`.

In [ ]:
from rest_framework.pagination import PageNumberPagination


class ProductPagination(PageNumberPagination):
    page_size = 10
    page_query_param = "pagina"
    page_size_query_param = "por_pagina"
    max_page_size = 50

## Configuración

- `page_size = 10`: muestra 10 productos por página.
- `page_query_param = "pagina"`: permite seleccionar la página mediante el parámetro `pagina`.
- `page_size_query_param = "por_pagina"`: permite modificar el número de productos por página.
- `max_page_size = 50`: limita a 50 el máximo de productos que pueden mostrarse por página.

## Integración con ProductViewSet

In [ ]:
from rest_framework import status, viewsets
from rest_framework.response import Response

from .models import ProductModel
from .pagination import ProductPagination
from .serializers import ProductSerializer


class ProductViewSet(viewsets.ModelViewSet):
    queryset = ProductModel.objects.all().order_by("id")
    serializer_class = ProductSerializer
    pagination_class = ProductPagination

    def create(self, request, *args, **kwargs):
        serializer = self.get_serializer(
            data=request.data
        )
        serializer.is_valid(raise_exception=True)
        self.perform_create(serializer)

        return Response(
            serializer.data,
            status=status.HTTP_201_CREATED,
        )

    def list(self, request, *args, **kwargs):
        queryset = self.filter_queryset(
            self.get_queryset()
        )

        page = self.paginate_queryset(queryset)

        if page is not None:
            serializer = self.get_serializer(
                page,
                many=True,
            )

            return self.get_paginated_response(
                serializer.data
            )

        serializer = self.get_serializer(
            queryset,
            many=True,
        )

        return Response(serializer.data)

    def retrieve(self, request, *args, **kwargs):
        instance = self.get_object()
        serializer = self.get_serializer(instance)

        return Response(serializer.data)

    def update(self, request, *args, **kwargs):
        partial = kwargs.pop("partial", False)
        instance = self.get_object()

        serializer = self.get_serializer(
            instance,
            data=request.data,
            partial=partial,
        )

        serializer.is_valid(raise_exception=True)
        self.perform_update(serializer)

        return Response(serializer.data)

    def partial_update(
        self,
        request,
        *args,
        **kwargs,
    ):
        kwargs["partial"] = True

        return self.update(
            request,
            *args,
            **kwargs,
        )

    def destroy(self, request, *args, **kwargs):
        instance = self.get_object()
        self.perform_destroy(instance)

        return Response(
            {
                "message":
                "Producto eliminado correctamente"
            },
            status=status.HTTP_200_OK,
        )

# Pruebas realizadas

## Prueba 1: primera página

Se realizó la petición:

GET /api/viewset/products/?pagina=1

Resultado:

- Total de productos: 500
- Productos mostrados: 10
- `previous`: null
- `next`: página 2

Esto comprobó que el tamaño predeterminado de página es 10.

## Prueba 2: segunda página

Se realizó la petición:

GET /api/viewset/products/?pagina=2

Resultado:

- Se mostraron los productos 11 al 20.
- `previous` permitió regresar a la primera página.
- `next` apuntó a la página 3.

Esto comprobó el funcionamiento del parámetro personalizado
`pagina`.

## Prueba 3: tamaño personalizado

Se realizó la petición:

GET /api/viewset/products/?pagina=1&por_pagina=5

Resultado:

- Se devolvieron únicamente 5 productos.
- El enlace `next` conservó el parámetro `por_pagina=5`.

Esto comprobó el funcionamiento de `page_size_query_param`.

## Prueba 4: límite máximo

Se realizó la petición:

GET /api/viewset/products/?pagina=1&por_pagina=1000

Aunque se solicitaron 1000 productos, la API devolvió solamente
50 productos.

El último resultado fue Producto 50 con ID 51.

Esto comprobó que `max_page_size = 50` funciona correctamente.


# Conclusión

Se implementó correctamente `PageNumberPagination` en la API
del eCommerce.

La paginación permite navegar entre páginas mediante `pagina`,
modificar el tamaño mediante `por_pagina` y limita cada página
a un máximo de 50 productos.
